In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from xgboost import XGBClassifier
import optuna

In [ ]:
import numpy as np

In [ ]:
train_data = pd.read_parquet('train.parquet')
print(train_data.head())


In [ ]:
test_data = pd.read_parquet('test.parquet')
print(test_data.head(10))

In [ ]:
import pandas as pd

def clean_raw_data(raw_df):
    """
    Phase 1: Basic Cleaning & Type Conversion
    """
    df = raw_df.copy()

    df['event_time'] = pd.to_datetime(df['ts'], unit='ms')
    df['registration_time'] = pd.to_datetime(df['registration'], unit='ms')
    
    cols_to_drop = [
        'ts', 'time', 
        'firstName', 'lastName', 
        'auth', 'method',
        'registration' 
    ]
    existing_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=existing_drop)
    
    print(f"Shape: {df.shape}")
    print("Columns remaining:", df.columns.tolist())
    
    return df
clean_df = clean_raw_data(train_data)
print("\nTime Check:")
print(clean_df['event_time'].head())

In [ ]:
def prepare_events(df):
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]
    if 'event_time' not in df.columns:
        df['event_time'] = pd.to_datetime(df['ts'], unit='ms')
    else:
        df['event_time'] = pd.to_datetime(df['event_time'])
    if 'registration_time' not in df.columns:
        df['registration_time'] = pd.to_datetime(df['registration'], unit='ms')
    else:
        df['registration_time'] = pd.to_datetime(df['registration_time'])
    df = df.loc[df['userId'].notna() & df['userId'].astype(str).str.strip().ne('')]
    df = df.sort_values(['userId', 'event_time'])
    return df


Event Features

In [ ]:
def add_event_features(df):
    df['is_song'] = (df['page'] == 'NextSong').astype(int)
    df['is_error'] = (df['status'] == 404).astype(int)
    df['is_thumbs_down'] = (df['page'] == 'Thumbs Down').astype(int)
    df['is_paid_event'] = (df['level'] == 'paid').astype(int)
    df['is_ad'] = (df['page'] == 'Roll Advert').astype(int)

    df['next_event_time'] = df.groupby('userId')['event_time'].shift(-1)
    df['actual_duration'] = (df['next_event_time'] - df['event_time']).dt.total_seconds()
    df['length'] = pd.to_numeric(df['length'], errors='coerce')

    df['is_skipped'] = 0
    song_rows = df['is_song'] == 1
    df.loc[song_rows, 'is_skipped'] = np.where(
        df.loc[song_rows, 'actual_duration'] < (df.loc[song_rows, 'length'] - 10), 1, 0
    )

    df['session_duration'] = df.groupby(['userId','sessionId'])['event_time'].transform(lambda x: x.max() - x.min())
    return df


Grouping, pandas groupby

In [ ]:
def build_user_core(df):
    user_features = df.groupby('userId', as_index=False).agg({
        'registration_time': 'first',
        'event_time': 'max',
        'gender': 'first',
        'location': 'first',
        'userAgent': 'first',
        'is_song': 'sum',
        'is_skipped': 'sum',
        'is_error': 'sum',
        'is_thumbs_down': 'sum',
        'is_paid_event': 'mean',
        'page': 'count'
    }).rename(columns={
        'event_time': 'last_active_time',
        'is_song': 'total_songs',
        'page': 'total_actions',
        'is_paid_event': 'percent_paid_time'
    })

    active_time = (
        df[['userId','sessionId','session_duration']].drop_duplicates()
        .groupby('userId')['session_duration'].sum()
        .dt.total_seconds() / 60
    )
    active_time = active_time.rename('total_active_minutes').reset_index()
    return user_features.merge(active_time, on='userId', how='left')


In [ ]:
def add_page_and_song_stats(df, user_features):
    page_counts = pd.crosstab(df['userId'], df['page']).reset_index()
    for col in ['Roll Advert','Thumbs Down','Add to Playlist']:
        if col not in page_counts.columns: page_counts[col] = 0
    user_features = user_features.merge(
        page_counts[['userId','Roll Advert','Thumbs Down','Add to Playlist']],
        on='userId', how='left'
    )

    song_df = df[df['is_song'] == 1][['userId','actual_duration','length']].copy()
    song_df['length'] = pd.to_numeric(song_df['length'], errors='coerce')
    song_df['actual_duration'] = pd.to_numeric(song_df['actual_duration'], errors='coerce')
    song_df['song_percentage'] = (100 * (song_df['actual_duration'] / song_df['length'])).replace([np.inf,-np.inf], np.nan).clip(0,100)

    song_stats = song_df.groupby('userId')['song_percentage'].agg(
        song_percent_avg='mean', song_percent_std='std'
    ).fillna({'song_percent_std': 0}).reset_index()

    user_features = user_features.merge(song_stats, on='userId', how='left')
    user_features['total_active_minutes'] = user_features['total_active_minutes'].fillna(0)
    for col in ['song_percent_avg','song_percent_std']:
        if col not in user_features.columns: user_features[col] = 0
        user_features[col] = user_features[col].fillna(0)
    return user_features


In [ ]:
def add_window_and_rates(df, user_features, cutoff_time):
    def add_window_counts(window_days):
        w_start = cutoff_time - pd.Timedelta(days=window_days)
        df_w = df[df['event_time'] >= w_start]

        ev = df_w.groupby('userId').size()
        so = df_w[df_w['is_song'] == 1].groupby('userId').size()
        ad = df_w[df_w['is_ad'] == 1].groupby('userId').size()
        td = df_w[df_w['is_thumbs_down'] == 1].groupby('userId').size()

        user_features[f'events_last_{window_days}d'] = user_features['userId'].map(ev).fillna(0)
        user_features[f'songs_last_{window_days}d'] = user_features['userId'].map(so).fillna(0)
        user_features[f'ads_last_{window_days}d'] = user_features['userId'].map(ad).fillna(0)
        user_features[f'thumbs_down_last_{window_days}d'] = user_features['userId'].map(td).fillna(0)

    for w in [3, 7, 14]:
        add_window_counts(w)

    denom_songs = user_features['total_songs'] + 1
    denom_actions = user_features['total_actions'] + 1
    denom_mins = user_features['total_active_minutes'] + 1

    user_features['skip_rate'] = user_features['is_skipped'] / denom_songs
    user_features['404_per_minute'] = user_features['is_error'] / denom_mins
    user_features['thumbs_down_per_song'] = user_features['is_thumbs_down'] / denom_songs

    user_features['avg_minutes_per_action'] = user_features['total_active_minutes'] / denom_actions
    user_features['avg_minutes_per_song'] = user_features['total_active_minutes'] / denom_songs
    user_features['ads_per_hour'] = (user_features['Roll Advert'] / denom_mins) * 60
    user_features['thumbs_down_per_hour'] = (user_features['Thumbs Down'] / denom_mins) * 60
    user_features['playlist_per_hour'] = (user_features['Add to Playlist'] / denom_mins) * 60

    user_features['lifetime_days'] = ((cutoff_time - user_features['registration_time']).dt.total_seconds() / (24 * 3600)).clip(lower=0.1)
    user_features['days_since_last_event'] = ((cutoff_time - user_features['last_active_time']).dt.total_seconds() / (24 * 3600)).clip(lower=0)
    user_features['songs_per_day'] = user_features['total_songs'] / user_features['lifetime_days']

    previous_events = df[(df['event_time'] >= cutoff_time - pd.Timedelta(days=28)) &
                         (df['event_time'] < cutoff_time - pd.Timedelta(days=7))].groupby('userId').size()
    user_features['activity_trend'] = user_features['events_last_7d'] / (user_features['userId'].map(previous_events).fillna(0) + 1)
    return user_features


In [ ]:
def add_labels_and_encode(df, user_features, churn_users=None):
    if churn_users is None:
        churn_users = df.loc[df['page']=='Cancellation Confirmation','userId'].unique()
    user_features['churn'] = user_features['userId'].isin(churn_users).astype(int)
    user_features['is_male'] = (user_features['gender']=='M').astype(int)

    def get_state(loc_str):
        if not isinstance(loc_str, str): return 'Unknown'
        parts = loc_str.split(',')
        return parts[-1].strip()[:2] if len(parts) > 1 else 'Unknown'

    user_features['state'] = user_features['location'].apply(get_state)
    user_features = pd.get_dummies(user_features, columns=['state'], drop_first=True)

    def get_os(ua):
        if not isinstance(ua, str): return 'Other'
        if 'iPhone' in ua or 'iPad' in ua: return 'iOS'
        if 'Android' in ua: return 'Android'
        if 'Windows' in ua: return 'Windows'
        if 'Mac' in ua: return 'Mac'
        if 'Linux' in ua: return 'Linux'
        return 'Other'

    user_features['os'] = user_features['userAgent'].apply(get_os)
    user_features = pd.get_dummies(user_features, columns=['os'], drop_first=True)

    drop_cols = [
        'registration_time','last_active_time','gender','location','userAgent',
        'recent_activity_count','prev_activity_count',
        'Roll Advert','Thumbs Down','Add to Playlist'
    ]
    user_features = user_features.drop(columns=[c for c in drop_cols if c in user_features.columns])
    user_features = user_features.loc[:, ~user_features.columns.duplicated()]
    return user_features


In [ ]:
def process_data(clean_df, churn_users=None, cutoff_time=None):
    df = prepare_events(clean_df)
    if df.empty:
        return pd.DataFrame()
    if cutoff_time is None:
        cutoff_time = df['event_time'].max()
    cutoff_time = pd.to_datetime(cutoff_time)
    df = df.loc[df['event_time'] <= cutoff_time].copy()
    if df.empty:
        return pd.DataFrame()

    df = add_event_features(df)
    user_features = build_user_core(df)
    user_features = add_page_and_song_stats(df, user_features)
    user_features = add_window_and_rates(df, user_features, cutoff_time)
    user_features = add_labels_and_encode(df, user_features, churn_users)

    return user_features.set_index('userId').fillna(0)


In [ ]:
#building snapshots can help us handle time-based churn prediction without leakage

def build_snapshot(clean_df, cutoff, horizon_days=10):
    clean_df = clean_df.copy()
    cutoff = pd.to_datetime(cutoff)
    horizon_end = cutoff + pd.Timedelta(days=horizon_days)
    feature_df = clean_df[clean_df['event_time'] <= cutoff].copy()

    label_next = clean_df[(clean_df['event_time'] > cutoff) & (clean_df['event_time'] <= horizon_end)].copy()
    label_ever = clean_df[(clean_df['event_time'] > cutoff)].copy()

    churn_users_next = label_next.loc[label_next['page'] == 'Cancellation Confirmation', 'userId'].unique()
    churn_users_ever = label_ever.loc[label_ever['page'] == 'Cancellation Confirmation', 'userId'].unique()

    # users already churned by cutoff should be excluded
    already_churned = set(clean_df.loc[
        (clean_df['event_time'] <= cutoff) & (clean_df['page'] == 'Cancellation Confirmation'),
        'userId'
    ].unique())

    feature_users = set(feature_df['userId'].unique())
    churn_users_next = [u for u in churn_users_next if (u in feature_users and u not in already_churned)]
    churn_users_ever = [u for u in churn_users_ever if (u in feature_users and u not in already_churned)]
    
    snap = process_data(feature_df, cutoff_time=cutoff, churn_users=churn_users_next)
    if snap is None or snap.empty:
        return pd.DataFrame()
    snap = snap.copy()
    snap['userId'] = snap.index
    snap = snap.reset_index(drop=True)
    snap['churn_ever'] = snap['userId'].isin(set(churn_users_ever)).astype(int)
    snap['cutoff_time'] = cutoff
    snap = snap[~snap['userId'].isin(already_churned)].copy()

    return snap


In [ ]:
def build_rolling_dataset(clean_df, horizon_days=10, step_days=7, min_history_days=14):
    clean_df = clean_df.copy()

    if 'event_time' not in clean_df.columns:
        clean_df['event_time'] = pd.to_datetime(clean_df['ts'], unit='ms')
    clean_df['event_time'] = pd.to_datetime(clean_df['event_time'])

    if 'registration_time' not in clean_df.columns:
        if 'registration' in clean_df.columns:
            clean_df['registration_time'] = pd.to_datetime(clean_df['registration'], unit='ms')
        else:
            raise KeyError("Missing both 'registration_time' and 'registration'.")

    start_time = clean_df['event_time'].min()
    end_time   = clean_df['event_time'].max()

    last_cutoff  = end_time - pd.Timedelta(days=horizon_days)
    first_cutoff = start_time + pd.Timedelta(days=min_history_days)

    if first_cutoff > last_cutoff:
        raise ValueError("Not enough time range for these min_history_days / horizon_days.")

    cutoffs = pd.date_range(first_cutoff, last_cutoff, freq=f'{step_days}D')

    snapshots = []
    for c in cutoffs:
        print(f"\nBuilding snapshot at cutoff={c} ")
        snap = build_snapshot(clean_df, cutoff=c, horizon_days=horizon_days)
        if not snap.empty:
            snapshots.append(snap)
    return pd.concat(snapshots, ignore_index=True)

clean_train = train_data.copy()
clean_train['event_time'] = pd.to_datetime(clean_train['ts'], unit='ms')
clean_train['registration_time'] = pd.to_datetime(clean_train['registration'], unit='ms')
final_train_df = build_rolling_dataset(clean_train, horizon_days=10, step_days=7, min_history_days=14)

print(final_train_df.shape)
print(final_train_df['churn'].value_counts(normalize=True))
print("Unique users:", final_train_df['userId'].nunique())
print("Median snapshots/user:", final_train_df.groupby('userId').size().median())

In [ ]:
final_train_df.columns 

In [ ]:
NON_FEATURE_COLS = ['churn', 'churn_ever', 'userId', 'cutoff_time']

X = final_train_df.drop(columns=[c for c in NON_FEATURE_COLS if c in final_train_df.columns], errors='ignore').copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
y_next = final_train_df['churn'].copy()
y_ever = final_train_df['churn_ever'].copy()

cutoffs_sorted = np.sort(final_train_df['cutoff_time'].unique())
n_test = max(1, int(np.ceil(0.2 * len(cutoffs_sorted))))
test_cutoffs = set(cutoffs_sorted[-n_test:])
test_start = pd.Timestamp(min(test_cutoffs))
is_test = final_train_df['cutoff_time'].isin(test_cutoffs)
is_train = (~is_test) & (final_train_df['cutoff_time'] + pd.Timedelta(days=10) <= test_start)

X_train, X_test = X.loc[is_train].copy(), X.loc[is_test].copy()
y_next_train, y_next_test = y_next.loc[is_train].copy(), y_next.loc[is_test].copy()
y_ever_test = y_ever.loc[is_test].copy()
groups_train = final_train_df.loc[is_train, 'userId']

prior_cancellations = clean_train.loc[
    (clean_train['page'] == 'Cancellation Confirmation') &
    (clean_train['event_time'] < test_start)
].groupby('userId')['event_time'].min()
train_cutoffs = final_train_df.loc[is_train, 'cutoff_time']
train_cancellations = final_train_df.loc[is_train, 'userId'].map(prior_cancellations)
y_ever_train = ((train_cancellations > train_cutoffs) & (train_cancellations < test_start)).astype(int)

print('Train rows:', X_train.shape, 'Test rows:', X_test.shape)
print('Next churn rate (train):', y_next_train.mean())
print('Ever churn rate (train):', y_ever_train.mean())

binary_cols = [c for c in X_train.columns if X_train[c].dropna().nunique() <= 2]
numeric_cols = [c for c in X_train.columns if c not in binary_cols]


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone

class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None, lower=0.01, upper=0.99):
        self.columns = columns
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        frame = pd.DataFrame(X)
        cols = list(self.columns) if self.columns is not None else frame.columns.tolist()
        self.columns_ = cols
        self.lower_bounds_ = frame[cols].quantile(self.lower)
        self.upper_bounds_ = frame[cols].quantile(self.upper)
        return self

    def transform(self, X):
        frame = pd.DataFrame(X).copy()
        frame.loc[:, self.columns_] = frame[self.columns_].clip(self.lower_bounds_, self.upper_bounds_, axis=1)
        return frame

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())]), numeric_cols),
        ('bin', SimpleImputer(strategy='constant', fill_value=0), binary_cols)
    ]
)

def class_ratio(y):
    return float((y == 0).sum()) / max(int((y == 1).sum()), 1)

base_xgb_next = XGBClassifier(
    scale_pos_weight=class_ratio(y_next_train),
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
base_xgb_ever = XGBClassifier(
    scale_pos_weight=class_ratio(y_ever_train),
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

tune = True
n_trials = 100 if tune else 1
cv_folds = 5 if tune else 2
gkf = GroupKFold(n_splits=cv_folds)

def suggest_xgb_params(trial):
    return {
        'model__n_estimators': trial.suggest_categorical('model__n_estimators', [100, 200, 400, 600, 800]),
        'model__max_depth': trial.suggest_categorical('model__max_depth', [5, 7, 9, 11]),
        'model__learning_rate': trial.suggest_categorical('model__learning_rate', [0.05, 0.1, 0.15, 0.2]),
        'model__subsample': trial.suggest_categorical('model__subsample', [0.7, 0.8, 0.9, 1.0]),
        'model__colsample_bytree': trial.suggest_categorical('model__colsample_bytree', [0.7, 0.8, 0.9, 1.0]),
        'model__min_child_weight': trial.suggest_categorical('model__min_child_weight', [1, 3, 5]),
        'model__gamma': trial.suggest_categorical('model__gamma', [0, 0.5, 1]),
        'model__booster': 'gbtree'
    }

def build_pipeline(base_model):
    return Pipeline([
        ('clipper', QuantileClipper(columns=numeric_cols)),
        ('preprocessor', clone(preprocessor)),
        ('model', clone(base_model))
    ])

def group_cv_balanced_accuracy(pipe, X, y, groups, splitter):
    scores = []
    for tr_idx, va_idx in splitter.split(X, y, groups):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        fitted = clone(pipe)
        fitted.set_params(model__scale_pos_weight=class_ratio(y_tr))
        fitted.fit(X_tr, y_tr)
        scores.append(balanced_accuracy_score(y_va, fitted.predict(X_va)))
    return float(np.mean(scores))

def tune_pipeline_with_optuna(X, y, groups, base_model, study_name='xgb'):
    pipe = build_pipeline(base_model)
    if not tune:
        pipe.fit(X, y)
        return pipe, {}

    def objective(trial):
        params = suggest_xgb_params(trial)
        model = clone(pipe).set_params(**params)
        return group_cv_balanced_accuracy(model, X, y, groups, gkf)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        study_name=study_name
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_pipe = clone(pipe).set_params(**study.best_params)
    best_pipe.fit(X, y)
    return best_pipe, study.best_params

best_next, best_params_next = tune_pipeline_with_optuna(
    X_train, y_next_train, groups_train, base_xgb_next, study_name='xgb_next'
)
print('Best NEXT params:', best_params_next)

best_ever, best_params_ever = tune_pipeline_with_optuna(
    X_train, y_ever_train, groups_train, base_xgb_ever, study_name='xgb_ever'
)
print('Best EVER params:', best_params_ever)

p_next = best_next.predict_proba(X_test)[:, 1]
p_ever = best_ever.predict_proba(X_test)[:, 1]


In [ ]:
from sklearn.base import clone

def out_of_fold_probabilities(model, X, y, groups, splitter):
    probabilities = np.full(len(X), np.nan)
    for tr_idx, va_idx in splitter.split(X, y, groups):
        fitted = clone(model)
        fitted.set_params(model__scale_pos_weight=class_ratio(y.iloc[tr_idx]))
        fitted.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        probabilities[va_idx] = fitted.predict_proba(X.iloc[va_idx])[:, 1]
    return probabilities

p_next_oof = out_of_fold_probabilities(best_next, X_train, y_next_train, groups_train, gkf)
p_ever_oof = out_of_fold_probabilities(best_ever, X_train, y_ever_train, groups_train, gkf)

threshold = 0.41
alpha_grid = np.linspace(0, 1, 101)
blend_scores = [balanced_accuracy_score(
    y_next_train,
    (alpha * p_next_oof + (1 - alpha) * p_ever_oof > threshold).astype(int)
) for alpha in alpha_grid]
best_alpha_grid = float(alpha_grid[int(np.argmax(blend_scores))])

print(f'Best alpha via grouped out-of-fold predictions: {best_alpha_grid:.2f}')
print(f'Best CV Balanced Accuracy: {max(blend_scores):.4f}')

p_blend_test = best_alpha_grid * p_next + (1 - best_alpha_grid) * p_ever
y_pred_test = (p_blend_test > threshold).astype(int)


In [ ]:
roc_auc = roc_auc_score(y_next_test, p_blend_test)
f1 = f1_score(y_next_test, y_pred_test, zero_division=0)
balanced_acc = balanced_accuracy_score(y_next_test, y_pred_test)

print(f'Balanced Accuracy: {balanced_acc:.4f}')
print(f'ROC AUC: {roc_auc:.4f}')
print(f'F1: {f1:.4f}')


In [ ]:
X_full = X.copy()
y_next_full = final_train_df['churn'].copy()
y_ever_full = final_train_df['churn_ever'].copy()

best_next.set_params(model__scale_pos_weight=class_ratio(y_next_full))
best_ever.set_params(model__scale_pos_weight=class_ratio(y_ever_full))
best_next.fit(X_full, y_next_full)
best_ever.fit(X_full, y_ever_full)


In [ ]:
test_raw_clean = test_data.copy()
test_raw_clean['event_time'] = pd.to_datetime(test_raw_clean['ts'], unit='ms')
test_raw_clean['registration_time'] = pd.to_datetime(test_raw_clean['registration'], unit='ms')

test_features = process_data(test_raw_clean)
test_user_ids = test_features.index

X_test_feat = test_features.drop(columns=[c for c in NON_FEATURE_COLS if c in test_features.columns], errors='ignore').copy()
X_test_feat = X_test_feat.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)

missing_cols = [c for c in X_full.columns if c not in X_test_feat.columns]
for c in missing_cols:
    X_test_feat[c] = 0

extra_cols = [c for c in X_test_feat.columns if c not in X_full.columns]
if extra_cols:
    X_test_feat = X_test_feat.drop(columns=extra_cols)

X_test_final = X_test_feat[X_full.columns].copy()

print(f'Train Shape: {X_full.shape}')
print(f'Test Shape:  {X_test_final.shape}')


In [ ]:
p_next_test = best_next.predict_proba(X_test_final)[:, 1]
p_ever_test = best_ever.predict_proba(X_test_final)[:, 1]

p_final = best_alpha_grid * p_next_test + (1 - best_alpha_grid) * p_ever_test
predictions = (p_final > threshold).astype(int)

submission = pd.DataFrame({'id': test_user_ids, 'target': predictions})
submission.to_csv('submission_hybrid.csv', index=False)
print('Saved submission_hybrid.csv')
print(submission['target'].value_counts())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

def plot_roc_blended(y_true, p_next, p_ever, alpha):
    p_blend = alpha * p_next + (1 - alpha) * p_ever

    fpr, tpr, _ = roc_curve(y_true, p_blend)
    roc_auc = auc(fpr, tpr)

    fig, ax = plt.subplots(figsize=(5.5, 4))
    ax.plot(fpr, tpr, linewidth=2, label=f"Blended ROC (AUC = {roc_auc:.4f})")
    ax.plot([0, 1], [0, 1], linewidth=1, linestyle="--", label="Random")

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve (Final Blended Model)")
    ax.legend(loc="lower right")

    plt.tight_layout()
    return fig, roc_auc

fig, roc_auc = plot_roc_blended(y_next_test, p_next, p_ever, best_alpha_grid)
plt.show()

fig.savefig("fig_roc_blended.png", dpi=300, bbox_inches="tight")
print("AUC:", roc_auc)
